In [3]:
import os
import copy
import ipynbname
import pandas as pd
import numpy as np
from Functions.AutoCloud_V6 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT_R2 import *
from Functions.TedaOptimize_V6 import *      

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.expand_frame_repr', False) # never wrap into a second block
pd.set_option('display.max_rows', 500)    # trunca só acima de 500 linhas

FileName = ipynbname.name()

RS = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')
sig = HI['PC1'].values
PARAMS_PATH = r'Optimization\TedaOpt_R6_R1_B1_1-B1_2_restart_symmetric_G0P0\mono\Optimization.csv'
PARAMS_PATH = r'Optimization\TedaOpt_R7_T2_B1_1-B1_2_restart_symmetric_G0P0\mono\Optimization.csv'
df = pd.read_csv(PARAMS_PATH).iloc[:,:]
dfs = df.sort_values(df.columns[-4], ascending=True)
EOL = 0.3

# rho do intervalo do HI (protocolo 3): RHO = valor estático ou inicial; RHO_KW = modo e parâmetros (AutoCloud_V6)
#   'static'    : rho constante (= RHO)
#   'grid'      : argmax V = média(cov)*média(sp) sobre a grade rho_n (rho_min..rho_max), esquecimento rho_lam
#   'grad'      : subida de gradiente (rho_eta, rho_T, rho_beta)
#   'conformal' : rho <- rho*(1 + rho_kappa*(miss - rho_alpha)), cobertura ~ 1 - rho_alpha
RHO = 0.1
#RHO_KW = dict(rho_mode='grid', rho_min=1e-3, rho_max=10.0, rho_n=50, rho_lam=1.0)
RHO_KW = dict(rho_mode='static')
#RHO_KW = dict(rho_mode='grad', rho_eta=0.2, rho_T=0.3, rho_beta=0.0)
#RHO_KW = dict(rho_mode='conformal', rho_kappa=0.05, rho_alpha=0.1)
MODE_RUL = 'interval'   # 'interval': HI e RUL intervalares (usa o rho); 'single': apenas pontuais (rho sem efeito)

# Treino em malha fechada das RNNs (AutoCloud_V6 / eRNN_R5): cl_H = 0 desliga (resultados idênticos ao Teda_R4)
#   cl_H    : nº máximo de passos da repetição em previsão livre após cada fit
#   cl_eta  : multiplicador das taxas η na repetição;  cl_every: repetição a cada cl_every passos
CL_KW = dict(cl_H=0, cl_eta=1.0, cl_every=1)
CL_KW = dict(cl_H=10, cl_eta=0.3, cl_every=1)


In [ ]:
print(df[:100])

In [4]:
i = 25
params = df.iloc[i, 1:12].values
m,nI,nR,nO,mO,N1,N2,N3,τ,act,W = ParseParams(params)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 
m = 2.15
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry',**RHO_KW,**CL_KW
			  )

teda.SimGranularRUL(J,L,X,Y,Z,MODE_RUL)
teda.HI_Granularity()
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | "
      f"rho: {teda.rho_hist[0]:.3f} -> {teda.rho_hist[-1]:.3f} ({teda.rho_mode})")

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

HI : cov=0.528 | sp=0.969 | V=média(cov*sp)=0.5126 | Vsum=55.3581 | rho: 0.100 -> 0.100 (static)


In [ ]:
i = 25

RS1 = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')
sig = HI['PC1'].values
params = df.iloc[i, 1:12].values
m,nI,nR,nO,mO,N1,N2,N3,τ,act,W = ParseParams(params)
X, Y, Z, kX = PrepareData(RS1, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry',**RHO_KW,**CL_KW
			  )

teda.SimGranularRUL(J,L,X,Y,Z,MODE_RUL)
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | "
      f"rho: {teda.rho_hist[0]:.3f} -> {teda.rho_hist[-1]:.3f} ({teda.rho_mode})")

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

RS2 = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')
sig = HI['PC1'].values
X, Y, Z, kX = PrepareData(RS2, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 

teda.restart(ref,'keep',Y[0])
teda.SimGranularRUL(J,L,X,Y,Z,MODE_RUL)
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | "
      f"rho: {teda.rho_hist[0]:.3f} -> {teda.rho_hist[-1]:.3f} ({teda.rho_mode})")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')



In [ ]:
# Comparação dos modos de rho na sequência B1_1 -> B1_2 (mesmos parâmetros da linha i; predições pontuais iguais
# em todos os modos: só os intervalos mudam). rho dos modos dinâmicos começa em RHO e continua no 2º rolamento.
import matplotlib.pyplot as plt
data = [('B1_1', RS1, pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')['PC1']),
        ('B1_2', RS2, pd.read_csv(r'Dataset\HI1\Bearing1_2.csv')['PC1'])]
modos = {'static':    dict(rho_mode='static'),
         'grid':      dict(rho_mode='grid', rho_n=50, rho_lam=1.0),
         'grad':      dict(rho_mode='grad', rho_eta=0.2, rho_T=0.3, rho_beta=0.0),
         'conformal': dict(rho_mode='conformal', rho_kappa=0.05, rho_alpha=0.1)}
linhas, rhos = [], {}
for nome, kw in modos.items():
    def on_end(t, nm, k, nome=nome):
        rhos[(nome, nm)] = t.rho_hist.copy()
    _, rows = SimSeq([m,nI,nR,nO,mO,N1,N2,N3,τ,act], data, state='keep', mode_rul='interval', rho=RHO,
                     intr_state='carry', eol=EOL, w=W, on_end=on_end, rho_kw=kw, cl_kw=CL_KW)
    linhas += [dict(modo=nome, **r) for r in rows]
print(pd.DataFrame(linhas).drop(columns=['ordem']).to_string(index=False))

fig, axs = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, nm in zip(axs, ['B1_1', 'B1_2']):
    for nome in modos:
        ax.plot(rhos[(nome, nm)], label=nome)
    ax.set_title(f'{nm}: rho usado em cada passo'); ax.set_xlabel('passo'); ax.set_yscale('log'); ax.grid(alpha=0.3)
axs[0].set_ylabel('rho'); axs[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
brngs = [f'B1_{i+1}' for i in range(5)]
brngs = ['B1_1','B1_2','B1_3','B1_4','B1_5','B2_1','B2_2','B2_4',][:2]
paths = [f'Dataset\DSIR1n\{brng}.csv' for brng in brngs]
DSIs = [pd.read_csv(path) for path in paths]
#DSIs = [DSI.drop(["FTF"], axis=1) for DSI in DSIs]
ftrs = len(DSIs[0].columns)
DSIs = [RS1,RS2]
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry',**RHO_KW,**CL_KW
			  )
w = 1
for DSI in DSIs: 
    teda.SimGranulation(WindowDSI(DSI[19:],w))

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f}")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=ftrs,dsi_mode='hide')

In [ ]:
RS1 = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')

sig = HI['PC1'].values
# Melhor linha (menor 1ª coluna) do Optimization.csv de PARAMS_PATH, com a janela w da mesma linha
(m,nI,nR,nO,mO,N1,N2,N3,τ,act), W = BestParamsW(PARAMS_PATH)
X1, Y, Z, kX = PrepareData(RS1, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry',**RHO_KW,**CL_KW
			  )

teda.SimGranularRUL(J,L,X1,Y,Z,MODE_RUL)
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | "
      f"rho: {teda.rho_hist[0]:.3f} -> {teda.rho_hist[-1]:.3f} ({teda.rho_mode})")

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

RS2 = pd.read_csv(r'Dataset\DSIR1n\B1_2.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_2.csv')
sig = HI['PC1'].values
X2, Y, Z, kX = PrepareData(RS2, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 

teda.restart(ref,'keep',Y[0])
teda.SimGranularRUL(J,L,X2,Y,Z,MODE_RUL)
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | "
      f"rho: {teda.rho_hist[0]:.3f} -> {teda.rho_hist[-1]:.3f} ({teda.rho_mode})")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

In [ ]:
wR = (0.101690 - 0)/(0.3 - 0) 
wH = (0.013605	 - 0)/(0.2 - 0) 
print(wR, wH, wR+wH, wR*wH)

wR = (0.575803	 - 0)/(0.3 - 0) 
wH = (0.003675	 - 0)/(0.2 - 0) 
print(wR, wH, wR+wH, wR*wH)

In [ ]:
wR = (0.226344 - 0)/(0.3 - 0) 
wH = (0.001195		 - 0)/(0.2 - 0) 
print(wR, wH,wR+wH, wR*wH)

wR = (0.224894 - 0)/(0.3 - 0) 
wH = (0.000693	 - 0)/(0.2 - 0) 
print(wR, wH,wR+wH, wR*wH)

In [ ]:
path = PARAMS_PATH
df = pd.read_csv(path)
df = df.sort_values(df.columns[0], ascending=True)
df

$$\text{input: } Y_k = [h_{k-5},...,h_{k}]$$
$$\text{output: } \hat{Z}_k = [\hat{h}_{k-4},...,\hat{h}_{k+1},...,\hat{h}_{k+nO}]$$
$$\text{target: }{Z}_k = [{h}_{k-4},...,{h}_{k+1},...,{h}_{k+nO}]$$

input $$Y_k = [h_{k-5},...,h_{k}]$$
output $$\hat{Z}_k = [...,\hat{h}_{k+1},...,\hat{h}_{k+nO}]$$
target $${Z}_k = [...,{h}_{k+1},...,{h}_{k+nO}]$$